# Import

In [1]:
# Run once, then restart the Python session
# !pip install -U datasets huggingface_hub fsspec

import numpy as np
import matplotlib.pyplot as plt
import torch
import textwrap

from transformers import AutoModelForCausalLM, GPT2Tokenizer
from datasets import load_dataset

# Load pretrained GPT-2 model and tokenizer
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Hyperparameters
seq_len = 256
batch_size = 32

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

# Qualitative Assessment (Pre)

In [2]:
# Qualitative assessment
prompt = 'QUESTION: Would it be strange to have a pet rock and feed it styrofoam?'
# prompt = 'QUESTION: Where did the word "butterfly" come from?'

in2gpt = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0).to(device)

output = gpt2.generate(
    in2gpt,
    max_length=100,
    pad_token_id=50256,
    do_sample=True
)

print(tokenizer.decode(in2gpt[0].cpu()), '\n')
print(textwrap.fill(tokenizer.decode(output[0][len(in2gpt[0]):]), 60))

QUESTION: Would it be strange to have a pet rock and feed it styrofoam? 

   LIKEL: I like to see my pet rock feed. We both like to
make our own rocks and feed them as they are made, but I
will give your pet rock and feed it in
styrofoam.<|endoftext|>


# Text Data

In [3]:
# https://huggingface.co/datasets/THUDM/webglm-qa
dataset = load_dataset("THUDM/webglm-qa")
dataset

dataset["train"][3]

# My minor modification
txt = (
    "QUESTION: " + dataset["train"][0]["question"]
    + " ANSWER: " + dataset["train"][0]["answer"]
)
txt

README.md:   0%|          | 0.00/4.27k [00:00<?, ?B/s]

data/train.jsonl: reconstructing file:   0%|          |  0.00B /  115MB            

data/train.jsonl: downloading bytes:           |  0.00B            

data/validation.jsonl: reconstructing file:   0%|          |  0.00B / 2.63MB            

data/validation.jsonl: downloading bytes:           |  0.00B            

...

data/test.jsonl: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

'QUESTION: in football whats the point of wasting the first two plays with a rush - up the middle - not regular rush plays i get those ANSWER: The point of wasting the first two plays with a rush up the middle is to take advantage of the 40 second play clock and the two minute warning in professional and college football games. By running the ball directly up the middle, the offense hopes to be able to block everyone up and the running back can make a person miss and end up in the end zone[2]. Running up the middle is also the shortest path to the end zone, so it is often designed to move defensive players out of the way[3]. Additionally, running the ball up the middle enough times will cause linebackers and defensive backs to run toward the line of scrimmage, creating the perfect opportunity to throw the ball over their heads or in vacated areas[4]. This strategy is used to keep possession of the ball as long as possible rather than attempting to score[2][5].'

In [4]:
from IPython.display import Markdown, display

idx = np.random.randint(len(dataset['train']))

Markdown(display(dataset['train'][idx]['question'])),
print('\n')
Markdown(display(dataset['train'][idx]['answer']))

'Why people complain about being bumped to a higher tax bracket.'

'People complain about moving up to a higher tax bracket because they think that every dollar they earn is taxed at the higher rate, which would lead to lower take-home pay overall[4]. However, this is not true; when you move up to a higher tax bracket, only the income that is within that tax bracket range will be taxed at the higher rate[2]. In reality, moving up to a higher tax bracket is more good news than bad, as it indicates that they are making more money and have a net gain despite the fact that they are paying higher taxes[5].'

<IPython.core.display.Markdown object>

# Tokenize

In [5]:
# Tokenize the data
trainTokens = []

# Just the first 5k samples (~800k tokens) for speed and simplicity
for idx in range(5000):  # len(dataset['train']):

    # Get text from this example
    txt = f"QUESTION: {dataset['train'][idx]['question']} ANSWER: {dataset['train'][idx]['answer']}"

    # Tokenize and concatenate it
    trainTokens += tokenizer.encode(txt)

# Needs to be a torch tensor
trainTokens = torch.tensor(trainTokens)
trainTokens.shape

torch.Size([782659])

# Check

In [6]:
# Check a random batch
ix = torch.randint(len(trainTokens) - seq_len, size=(batch_size,))
X = trainTokens[ix[:, None] + torch.arange(seq_len)]

print(X.shape)

for t in X[:5]:
    print(
        f"*** Example:\n",
        textwrap.fill(tokenizer.decode(t), 75),
        "\n"
    )

torch.Size([32, 256])
*** Example:
 iating all the single instruments or vocals of a song[2], they may have an
approximate perception of what the music sounds like[1][2]. There have also
been studies that have looked at how some animals react to music, either
created by humans or themselves, and have found diverse
reactions[5].QUESTION: Explain e=mc^2. ANSWER: E=mc^2 is an equation
derived by the twentieth-century physicist Albert Einstein, in which E
represents units of energy, m represents units of mass, and c2 is the speed
of light squared[2][5]. According to the special theory of Relativity
(1905), we can get the infinite quantity of energy from a body that has
infinite mass[2][5]. The equation states that the kinetic energy (E) of a
body is equal to the increased relativistic mass (m) of the body times the
speed of light squared (c2)[2][5]. This equation can also be expressed as E
k = m c 2 ( 1 1 − v 2 c 2 − 1 ) {\displaystyle E_{k}=mc^{2}\left({\frac
{1}{\sqrt {1-{\frac {v^{2}}{c

# Fine-tune the model

In [7]:
# Move the model to the GPU
gpt2 = gpt2.to(device)

# Create the optimizer
optimizer = torch.optim.AdamW(
    gpt2.parameters(),
    lr=5e-5,
    weight_decay=0.01
)

In [ ]:
# Training takes ~8 mins on A100
num_samples = 1234

# Initialize losses
train_loss = np.zeros(num_samples)

for sampli in range(num_samples):

    # Get a batch of data
    ix = torch.randint(
        len(trainTokens) - seq_len,
        size=(batch_size,)
    )

    X = trainTokens[
        ix[:, None] + torch.arange(seq_len)
    ].to(device)

    # Forward pass
    gpt2.zero_grad()
    outputs = gpt2(X, labels=X)
    loss = outputs.loss

    # Backprop
    loss.backward()
    optimizer.step()

    # Store the per-sample loss
    train_loss[sampli] = loss.item()

    # Update progress display
    if sampli % 77 == 0:
        print(
            f"Sample {sampli:4}/{num_samples}, "
            f"train loss: {train_loss[sampli]:.4f}"
        )

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Sample    0/1234, train loss: 3.1607


# Loss

In [ ]:
# Plot the losses
plt.figure(figsize=(8, 3))
plt.plot(train_loss, 'k')

plt.gca().set(
    xlim=[0, num_samples + 1],
    xlabel='Data sample',
    ylabel='Loss',
    title='Train loss'
)

plt.show()

# Qualitative Assessment

In [ ]:
# Qualitative assessment
prompt = 'QUESTION: Would it be strange to have a pet rock and feed it styrofoam?'
# prompt = 'QUESTION: Where did the word "butterfly" come from?'

in2gpt = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0).to(device)

output = gpt2.generate(
    in2gpt,
    max_length=100,
    pad_token_id=50256,
    do_sample=True
)

print(tokenizer.decode(in2gpt[0].cpu()), '\n')
print(textwrap.fill(tokenizer.decode(output[0][len(in2gpt[0]):]), 60))